In [ ]:
# preparation and initial imports

# mount data and code directory on Drive
from google.colab import drive
drive.mount('/content/drive')

import sys
path = '/content/drive/My Drive/Colab Notebooks/'
sys.path.append(path)

import persuasion_utils as util

!pip install langchain langchain-openai langchain-anthropic langchain-qwen3 langchain-google-genai

with_arguments = True

gemini_api_key = "redacted"
openai_api_key = "redacted"
anthropic_api_key = "redacted"


In [ ]:
import csv

scenarios = []

scenario_labels = ['full_summary', 'party_1', 'party_2', 'facts', 'party_1_arguments', 'party_2_arguments', 'legal_issue' ]

# read the scenarios
with open(path+"scenarios.csv", 'r') as f:
  reader = csv.reader(f)

  for line in reader:

    scenario = {}

    for i in range(len(scenario_labels)):
        scenario[ scenario_labels[i] ] = line[i]
    scenarios.append( scenario )


In [ ]:
# load the prompts and init outputs

prompts = []

if with_arguments:
  for s in scenarios:
    for _ in range(5):
      prompts.append( util.generate_argument_prompt_with_arguments( s['party_1'], s['facts'], s['legal_issue'], s['party_1_arguments'], s['party_2_arguments'] ) )
    for _ in range(5):
      prompts.append( util.generate_argument_prompt_with_arguments( s['party_2'], s['facts'], s['legal_issue'], s['party_1_arguments'], s['party_2_arguments'] ) ) # present 1 first because of how they're written in the .csv
else:
  for s in scenarios:
    for _ in range(5):
      prompts.append( util.generate_argument_prompt( s['party_1'], s['facts'], s['legal_issue'] ) )
    for _ in range(5):
      prompts.append( util.generate_argument_prompt( s['party_2'], s['facts'], s['legal_issue'] ) )


output = [ [ 'Case_ID', 'Party_ID', 'Argument_ID', 'Model', 'Argument' ] ]


In [ ]:
# keep processing prompts until the list is empty (allows to restart from an interrupted session without losing the responses)

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic

total = len( prompts )

## ADVOCATE MODELS (uncomment as appropriate)

# model_string = "gemini-2.5-flash_reasoning"
model_string = "gemini-3-pro-preview_4096-thinking"
llm = ChatGoogleGenerativeAI(
    model=model_string[:model_string.index("_")],
    thinking_budget=4096,
    api_key=gemini_api_key"
)

# model_string = "claude-sonnet-4-5-20250929_8k-thinking"
# llm = ChatAnthropic(
#     model=model_string[:model_string.index("_")],
#     thinking={'type' : 'enabled', 'budget_tokens': 8192 },
#     max_tokens=16000,
#     api_key=anthropic_api_key
# )

# model_string = "gpt-5.1_low-reasoning"
# llm = ChatOpenAI(
#     model=model_string[:model_string.index("_")],
#     reasoning_effort="low", # "none" is not supported by this model (minimal/low/medium/high only)
#     api_key=openai_api_key
# )

# model_string = "gpt-4o_defaults"
# llm = ChatOpenAI(
#     model=model_string[:model_string.index("_")],
#     # reasoning_effort="low", # "none" is not supported by this model (minimal/low/medium/high only)
#     api_key=openai_api_key
# )

## END ADVOCATE MODELS

# Send each prompt individually
i = 0
while( len( prompts ) > 0 ):
    prompt = prompts[0]
    response = llm.invoke(prompt)
    # ( case_id(1...n), party_id(1,2), arg_id(1..5), model, content )
    output.append( [ ( i // 10 ) + 1, ( i // 5 % 2 ) + 1, ( i % 5 ) + 1, model_string, response.content ] )
    i += 1
    print( "Finished {} of {}".format( i, total ) )
    prompts.pop(0) #only remove from the list once it's complted successfully


with open( path + model_string + ( "_arguments.csv" if with_arguments else "_noarguments.csv" ), 'w' ) as f:
  writer = csv.writer(f)
  writer.writerows( output )